In [10]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import joblib

In [13]:
df = pd.read_csv("C:\\Users\\wasim\\Downloads\\customer_purchase_prediction_end_to_end.csv")

In [14]:
df.head()

,Customer_ID,Age,Gender,City,Device,Membership,Website_Visits,Pages_Viewed,Time_On_Site_Min,Previous_Purchases,Cart_Additions,Discount_Used,Email_Open_Rate,Customer_Rating,Annual_Income,Days_Since_Last_Purchase,Purchase
0,1001,56.0,Female,Mumbai,Laptop,Silver,26.0,14,24.34,14,0,No,0.51,3.0,41008.0,30,1
1,1002,46.0,Female,Noida,Laptop,Gold,6.0,15,9.64,0,4,Yes,0.46,2.2,57756.0,65,1
2,1003,32.0,Male,Bangalore,Laptop,Basic,23.0,14,12.47,13,1,No,0.63,3.8,95730.0,90,1
3,1004,60.0,Female,Delhi,Mobile,Gold,25.0,3,9.53,2,5,No,0.66,4.8,97702.0,68,1
4,1005,25.0,Male,Pune,Mobile,Gold,10.0,19,17.38,15,6,No,0.55,3.5,61840.0,164,1


In [15]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Value:")
print(df.isnull().sum())

Shape: (1000, 17)

Columns:
Index(['Customer_ID', 'Age', 'Gender', 'City', 'Device', 'Membership',
       'Website_Visits', 'Pages_Viewed', 'Time_On_Site_Min',
       'Previous_Purchases', 'Cart_Additions', 'Discount_Used',
       'Email_Open_Rate', 'Customer_Rating', 'Annual_Income',
       'Days_Since_Last_Purchase', 'Purchase'],
      dtype='str')

Data Types:
Customer_ID                   int64
Age                         float64
Gender                          str
City                            str
Device                          str
Membership                      str
Website_Visits              float64
Pages_Viewed                  int64
Time_On_Site_Min            float64
Previous_Purchases            int64
Cart_Additions                int64
Discount_Used                   str
Email_Open_Rate             float64
Customer_Rating             float64
Annual_Income               float64
Days_Since_Last_Purchase      int64
Purchase                      int64
dtype: object

Missing

In [16]:
df.describe

<bound method NDFrame.describe of      Customer_ID   Age  Gender       City  Device Membership  Website_Visits  \
0           1001  56.0  Female     Mumbai  Laptop     Silver            26.0   
1           1002  46.0  Female      Noida  Laptop       Gold             6.0   
2           1003  32.0    Male  Bangalore  Laptop      Basic            23.0   
3           1004  60.0  Female      Delhi  Mobile       Gold            25.0   
4           1005  25.0    Male       Pune  Mobile       Gold            10.0   
..           ...   ...     ...        ...     ...        ...             ...   
995         1996  50.0    Male      Noida  Tablet   Platinum            18.0   
996         1997  21.0    Male       Pune  Laptop      Basic            23.0   
997         1998  39.0    Male  Bangalore  Laptop     Silver            26.0   
998         1999  19.0  Female    Chennai  Laptop     Silver            17.0   
999         2000  27.0    Male  Hyderabad  Laptop     Silver             6.0   

     

In [18]:
print(df["Purchase"].value_counts())

Purchase
1    847
0    153
Name: count, dtype: int64


In [19]:
print(df["Purchase"].value_counts(normalize=True) *100)

Purchase
1    84.7
0    15.3
Name: proportion, dtype: float64


In [21]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [23]:
df = df.drop_duplicates()
print("New shape:", df.shape)

New shape: (1000, 17)


In [24]:
df.isnull().sum()

Customer_ID                  0
Age                         10
Gender                       0
City                         0
Device                      10
Membership                   0
Website_Visits              10
Pages_Viewed                 0
Time_On_Site_Min             0
Previous_Purchases           0
Cart_Additions               0
Discount_Used                0
Email_Open_Rate              0
Customer_Rating             10
Annual_Income               10
Days_Since_Last_Purchase     0
Purchase                     0
dtype: int64

In [26]:
df = df.drop(columns=["Customer_ID"])

In [27]:
df.head()

,Age,Gender,City,Device,Membership,Website_Visits,Pages_Viewed,Time_On_Site_Min,Previous_Purchases,Cart_Additions,Discount_Used,Email_Open_Rate,Customer_Rating,Annual_Income,Days_Since_Last_Purchase,Purchase
0,56.0,Female,Mumbai,Laptop,Silver,26.0,14,24.34,14,0,No,0.51,3.0,41008.0,30,1
1,46.0,Female,Noida,Laptop,Gold,6.0,15,9.64,0,4,Yes,0.46,2.2,57756.0,65,1
2,32.0,Male,Bangalore,Laptop,Basic,23.0,14,12.47,13,1,No,0.63,3.8,95730.0,90,1
3,60.0,Female,Delhi,Mobile,Gold,25.0,3,9.53,2,5,No,0.66,4.8,97702.0,68,1
4,25.0,Male,Pune,Mobile,Gold,10.0,19,17.38,15,6,No,0.55,3.5,61840.0,164,1


In [28]:
df["Visit_per_Page"] = (
    df["Website_Visits"] / 
    df["Pages_Viewed"].replace(0, np.nan)
)

In [30]:
df["Purchase_History_Score"] = (
    df["Previous_Purchases"] +
    df["Cart_Additions"] * 2
)

In [32]:
df["Engagement_Score"] = (
    df["Website_Visits"] +
    df["Pages_Viewed"] +
    df["Time_On_Site_Min"]
)

In [33]:
df.head()

,Age,Gender,City,Device,Membership,Website_Visits,Pages_Viewed,Time_On_Site_Min,Previous_Purchases,Cart_Additions,Discount_Used,Email_Open_Rate,Customer_Rating,Annual_Income,Days_Since_Last_Purchase,Purchase,Visit_per_Page,Purchase_History_Score,Engagement_Score
0,56.0,Female,Mumbai,Laptop,Silver,26.0,14,24.34,14,0,No,0.51,3.0,41008.0,30,1,1.857143,14,64.34
1,46.0,Female,Noida,Laptop,Gold,6.0,15,9.64,0,4,Yes,0.46,2.2,57756.0,65,1,0.400000,8,30.64
2,32.0,Male,Bangalore,Laptop,Basic,23.0,14,12.47,13,1,No,0.63,3.8,95730.0,90,1,1.642857,15,49.47
3,60.0,Female,Delhi,Mobile,Gold,25.0,3,9.53,2,5,No,0.66,4.8,97702.0,68,1,8.333333,12,37.53
4,25.0,Male,Pune,Mobile,Gold,10.0,19,17.38,15,6,No,0.55,3.5,61840.0,164,1,0.526316,27,46.38


In [34]:
X = df.drop(columns=["Purchase"])
y = df["Purchase"]

In [35]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 18)
y shape: (1000,)


In [38]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()
numeric_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()
print("Categorical:")
print(categorical_features)
print("\nNumerical:")
print(numeric_features)

Categorical:
['Gender', 'City', 'Device', 'Membership', 'Discount_Used']

Numerical:
['Age', 'Website_Visits', 'Pages_Viewed', 'Time_On_Site_Min', 'Previous_Purchases', 'Cart_Additions', 'Email_Open_Rate', 'Customer_Rating', 'Annual_Income', 'Days_Since_Last_Purchase', 'Visit_per_Page', 'Purchase_History_Score', 'Engagement_Score']


C:\Users\wasim\AppData\Local\Temp\ipykernel_416\1562421360.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [39]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("sclaer", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [41]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [43]:
print("Training data:", X_train.shape)
print("Training data:", X_test.shape)

Training data: (800, 18)
Training data: (200, 18)


In [45]:
logistic_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])
logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['Age','Gender','City',...,'Visit_per_Page','Purchase_History_Score', 'Engagement_Score']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

In [47]:
logistic_pred = logistic_model.predict(X_test)
logistic_probability = logistic_model.predict_proba(X_test)[:, 1]

In [52]:
print("Accuracy:",
      accuracy_score(y_test, logistic_pred))
print("Precision:",
      precision_score(y_test, logistic_pred))
print("Recall:",
      recall_score(y_test, logistic_pred))
print("F1:",
      f1_score(y_test, logistic_pred))
print("ROC-AUC:",
      roc_auc_score(y_test, logistic_probability))

Accuracy: 0.625
Precision: 0.8615384615384616
Recall: 0.6627218934911243
F1: 0.7491638795986622
ROC-AUC: 0.6569956098492079


In [62]:
random_forest_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
          n_estimators=250,
        max_depth=10,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ))
])
random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [63]:
rf_pred = random_forest_model.predict(X_test)

rf_probability = random_forest_model.predict_proba(
    X_test
)[:, 1]

In [64]:
print("Accuracy:",
      accuracy_score(y_test, logistic_pred))
print("Precision:",
      precision_score(y_test, logistic_pred))
print("Recall:",
      recall_score(y_test, logistic_pred))
print("F1:",
      f1_score(y_test, logistic_pred))
print("ROC-AUC:",
      roc_auc_score(y_test, logistic_probability))

Accuracy: 0.625
Precision: 0.8615384615384616
Recall: 0.6627218934911243
F1: 0.7491638795986622
ROC-AUC: 0.6569956098492079


In [65]:
gb_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    ))
])
gb_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['Age','Gender','City',...,'Visit_per_Page','Purchase_History_Score', 'Engagement_Score']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

In [68]:
gb_pred = gb_model.predict(X_test)
gb_probability = gb_model.predict_proba(X_test)[:, 1]

In [69]:
print("Accuracy:",
      accuracy_score(y_test, logistic_pred))
print("Precision:",
      precision_score(y_test, logistic_pred))
print("Recall:",
      recall_score(y_test, logistic_pred))
print("F1:",
      f1_score(y_test, logistic_pred))
print("ROC-AUC:",
      roc_auc_score(y_test, logistic_probability))


Accuracy: 0.625
Precision: 0.8615384615384616
Recall: 0.6627218934911243
F1: 0.7491638795986622
ROC-AUC: 0.6569956098492079


In [78]:
result = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Accuracy": [
       accuracy_score(y_test, logistic_pred),
       accuracy_score(y_test, rf_pred),
       accuracy_score(y_test, gb_pred)
    ],
   "Precision": [
      precision_score(y_test, logistic_pred),
      precision_score(y_test, rf_pred),
      precision_score(y_test, gb_pred)
   ],
 "Recall": [
     recall_score(y_test, logistic_pred),
     recall_score(y_test, rf_pred),
     recall_score(y_test, gb_pred)
 ],

 "F1": [
      f1_score(y_test, logistic_pred),
      f1_score(y_test, rf_pred),
      f1_score(y_test, gb_pred)
 ],

 "ROV_AUC": [
     roc_auc_score(y_test, logistic_pred),
     roc_auc_score(y_test, rf_pred),
     roc_auc_score(y_test, gb_pred)
 ]
})
result.sort_values(
    by="F1",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1,ROV_AUC
2,Gradient Boosting,0.815,0.851064,0.946746,0.896359,0.521760
1,Random Forest,0.790,0.858757,0.899408,0.878613,0.546478
0,Logistic Regression,0.625,0.861538,0.662722,0.749164,0.541038


In [80]:
rf_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [82]:
param_grid = {
    "model__n_estimators": [150, 200],
    "model__max_depth": [None, 10],
    "model__min_samples_split": [2, 5]
}

In [83]:
grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [None, 10], 'model__min_samples_split': [2, 5], 'model__n_estimators': [150, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, defa

In [84]:
print(grid_search.best_params_)

{'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 150}


In [85]:
final_model = grid_search.best_estimator_

In [86]:
final_pred = final_model.predict(X_test)
final_probabilty = final_model.predict_proba(X_test)[:, 1]

In [87]:
print("Accuracy:",
      accuracy_score(y_test, logistic_pred))
print("Precision:",
      precision_score(y_test, logistic_pred))
print("Recall:",
      recall_score(y_test, logistic_pred))
print("F1:",
      f1_score(y_test, logistic_pred))
print("ROC-AUC:",
      roc_auc_score(y_test, logistic_probability))

Accuracy: 0.625
Precision: 0.8615384615384616
Recall: 0.6627218934911243
F1: 0.7491638795986622
ROC-AUC: 0.6569956098492079


In [88]:
cm = confusion_matrix(
    y_test,
    final_pred
)
print(cm)

[[  6  25]
 [ 14 155]]


In [89]:
print(
    classification_report(
        y_test,
        final_pred
    )
)

              precision    recall  f1-score   support

           0       0.30      0.19      0.24        31
           1       0.86      0.92      0.89       169

    accuracy                           0.81       200
   macro avg       0.58      0.56      0.56       200
weighted avg       0.77      0.81      0.79       200



In [113]:
new_customer = pd.DataFrame([{
    "Age": 30,
    "Gender": "Male",
    "City": "Delhi",
    "Device": "Mobile",
    "Membership": "Gold",
    "Website_Visits": 15,
    "Pages_Viewed": 10,
    "Time_On_Site_Min": 12.5,
    "Previous_Purchases": 4,
    "Cart_Additions": 3,
    "Discount_Used": "Yes",
    "Email_Open_Rate": 0.70,
    "Customer_Rating": 4.3,
    "Annual_Income": 60000,
    "Days_Since_Last_Purchase": 20,
    "Visit_per_Page": 1.5,
    "Purchase_History_Score": 10,
    "Engagement_Score": 37.5
}])

In [114]:
new_customer["Visits_per_Page"] = (
    new_customer["Website_Visits"] /
    new_customer["Pages_Viewed"].replace(0, np.nan))
new_customer["Purchase_History_Score"] = (
    new_customer["Previous_Purchases"] +
    new_customer["Cart_Additions"] * 2)
new_customer["Engagement_Score"] = (
    new_customer["Website_Visits"] +
    new_customer["Pages_Viewed"] +
    new_customer["Time_On_Site_Min"])
print(new_customer.columns)

Index(['Age', 'Gender', 'City', 'Device', 'Membership', 'Website_Visits',
       'Pages_Viewed', 'Time_On_Site_Min', 'Previous_Purchases',
       'Cart_Additions', 'Discount_Used', 'Email_Open_Rate', 'Customer_Rating',
       'Annual_Income', 'Days_Since_Last_Purchase', 'Visit_per_Page',
       'Purchase_History_Score', 'Engagement_Score', 'Visits_per_Page'],
      dtype='str')


In [115]:
prediction = final_model.predict(new_customer)

# Probability
probability = final_model.predict_proba(new_customer)[0][1]

print("Prediction:", prediction[0])
print("Purchase Probability:", probability)

if prediction[0] == 1:
    print("Result: Customer is likely to purchase.")
else:
    print("Result: Customer is unlikely to purchase.")

Prediction: 1
Purchase Probability: 0.8549000933706816
Result: Customer is likely to purchase.


In [116]:
joblib.dump(
    final_model,
    "customer_purchase_prediction_model.pkl"
)
print("Model saved successfully!")

Model saved successfully!


In [117]:
loaded_model = joblib.load("customer_purchase_prediction_model.pkl")

In [118]:
loaded_prediction = loaded_model.predict(new_customer)
print(loaded_prediction)

[1]
